# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/h0pi/flyrank-ml-intenrship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup: same slice and label as week 3

Lane (locked this week): **Refresh / Content Opportunity Scoring**. Same data contract as `w03_data_contract.ipynb`, so the week-5 model can be compared on exactly the same rows and label:

- **Row:** one page with Google Search data in March 2026, seen at the decision moment **2026-03-16**.
- **Inputs:** only days **1–15 March** (plus the page's creation date).
- **Label (only for evaluation, never an input):** `is_declining` = average daily impressions on 16–31 March fell below 80% of 1–15 March.
- Filters: `gsc_data_available IS TRUE`, at least 100 impressions in 1–15 March, page created before March.

The first run reads the March partition from the release and caches a small per-page table in `work/outputs/` (parquet is git-ignored).

In [1]:

%pip -q install duckdb huggingface_hub

import os, json, getpass
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd

def hf_token():
    tok = os.environ.get("HF_TOKEN")
    if not tok:
        try:
            from google.colab import userdata
            tok = userdata.get("HF_TOKEN")
        except Exception:
            pass
    if not tok:
        try:
            from huggingface_hub import get_token
            tok = get_token()
        except Exception:
            pass
    return tok or getpass.getpass("Hugging Face READ token: ")

ROOT = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
OUT = ROOT / "work" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
CACHE = OUT / "march_pages.parquet"

if not CACHE.exists():
    con = duckdb.connect()
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token()}')")
    REL = "hf://datasets/FlyRank/internship-warehouse"
    con.sql(f"""
        WITH d AS (
            SELECT client_hash_id, content_hash_id, report_date, gsc_impressions, gsc_clicks, gsc_avg_position
            FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
            WHERE gsc_data_available IS TRUE
        ),
        p AS (
            SELECT client_hash_id, content_hash_id,
                   SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15')                            AS imp_h1,
                   SUM(gsc_clicks)      FILTER (WHERE report_date <= DATE '2026-03-15')                            AS clk_h1,
                   SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-07')                            AS imp_w1,
                   SUM(gsc_impressions) FILTER (WHERE report_date BETWEEN DATE '2026-03-08' AND DATE '2026-03-15') AS imp_w2,
                   SUM(gsc_avg_position * gsc_impressions)
                       FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_impressions > 0 AND gsc_avg_position > 0) AS pos_x_imp,
                   SUM(gsc_impressions)
                       FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_impressions > 0 AND gsc_avg_position > 0) AS imp_with_pos,
                   SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '2026-03-16')                            AS imp_h2
            FROM d GROUP BY 1, 2
            HAVING SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15') >= 100
        )
        SELECT p.*, c.content_created_date, c.content_type
        FROM p JOIN read_parquet('{REL}/dim_content.parquet') c USING (client_hash_id, content_hash_id)
        WHERE c.content_created_date < DATE '2026-03-01'
    """).df().to_parquet(CACHE)

df = pd.read_parquet(CACHE)
df["imp_h2"] = df["imp_h2"].fillna(0)
# inputs: knowable on 2026-03-16
df["imp_per_day"]  = df["imp_h1"] / 15
df["momentum"]     = np.log((df["imp_w2"].fillna(0) / 8 + 1) / (df["imp_w1"].fillna(0) / 7 + 1))
df["age_days"]     = (pd.Timestamp("2026-03-15") - pd.to_datetime(df["content_created_date"])).dt.days
df["avg_position"] = df["pos_x_imp"] / df["imp_with_pos"]
df["ctr"]          = df["clk_h1"] / df["imp_h1"]
# label: the future half of March, used ONLY to evaluate
df["is_declining"] = ((df["imp_h2"] / 16) < 0.8 * df["imp_per_day"]).astype(int)

BASE_RATE = df["is_declining"].mean()
print(f"pages: {len(df):,} from {df.client_hash_id.nunique()} clients | base rate (declining): {BASE_RATE:.1%}")


Note: you may need to restart the kernel to use updated packages.


pages: 74,377 from 36 clients | base rate (declining): 33.6%


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule in plain words:** *a page that is clearly visible in Google and has already started losing impressions in the first half of March should be reviewed first; the bigger and more visible the slide, the higher it goes.*

**Reason code (one):** `early_slide_visible`: impressions in 8–15 March are more than ~10% below 1–7 March (momentum < −0.1) on a page with at least 100 impressions in 1–15 March. Pages that don't match get `none`.

**Action label:** `review_page` for flagged pages, `monitor` for the rest.

Before encoding it, I check the two signals the rule could lean on. Each table shows the number of pages (`n`) and the share that declined in 16–31 March, next to the base rate.

### Signal A: staleness (behind FlyRank's refresh flags)

The refresh flags assume that **older, staler pages decay more**. The honest staleness field would be "days since last update", but `content_updated_date` is a July snapshot that can include edits made *after* 16 March (excluded in week 3 as leakage). So I use **page age at the decision moment** as the staleness proxy.

In [2]:

def bucket_table(col, bins, labels):
    b = pd.cut(df[col], bins=bins, labels=labels)
    t = df.groupby(b, observed=True)["is_declining"].agg(n="size", declining_share="mean")
    t["vs_base_rate"] = t["declining_share"] - BASE_RATE
    return t.round(3)

staleness = bucket_table("age_days", [0, 90, 180, 365, 10_000], ["< 90 days", "90–180", "180–365", "365+"])
staleness


,n,declining_share,vs_base_rate
age_days,,,
< 90 days,22057,0.353,0.017
90–180,15154,0.440,0.105
180–365,29407,0.286,-0.050
365+,7759,0.272,-0.064


**Verdict: MIXED.** Decline is highest for pages **90–180 days** old (44.0%, n = 15,154) and *lowest* for the oldest pages: 180–365 days 28.6% (n = 29,407) and 365+ days 27.2% (n = 7,759), both below the 33.6% base rate. So "older = decays more" holds only up to about six months and then flips. Past half a year it is the **opposite** of the refresh-flag logic.

What it means for my rule: age alone would push the oldest pages up, and they are the *least* likely to decline. I keep staleness **out of the score**. That clearly negative result just saved the rule from a bad input.

### Signal B: early momentum (first-half slide)

Momentum = log of (daily impressions 8–15 March ÷ daily impressions 1–7 March). Negative = the page was already sliding before the decision moment. Both weeks end before 16 March, so this is knowable at decision time.

In [3]:

momentum = bucket_table("momentum", [-99, -0.3, -0.1, 0.1, 0.3, 99],
                        ["falling hard (< −0.3)", "falling", "flat", "rising", "rising hard (> 0.3)"])
momentum


,n,declining_share,vs_base_rate
momentum,,,
falling hard (< −0.3),18531,0.537,0.201
falling,14482,0.323,-0.013
flat,15672,0.287,-0.049
rising,11225,0.253,-0.082
rising hard (> 0.3),14467,0.208,-0.128


**Verdict: CONFIRMED.** The decline share falls steadily from 53.7% for pages already falling hard (n = 18,531) to 20.8% for pages rising hard (n = 14,467), monotonic across all five buckets. A page that was already slipping in early March is far more likely to keep slipping. This is the signal the rule leans on.

(Side check, not one of the two: decline share is flat across impression-volume quartiles, 31–35%, so volume is used only to put *bigger* sliding pages first, not as a predictor.)

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Score = how hard the page slid × how visible it is (log of daily impressions, so one huge page can't dominate just by size). No fitted weights: every term is readable.

In [4]:

INPUTS = ["momentum", "imp_per_day"]          # the only columns the score is allowed to read

sliding = df["momentum"] < -0.1
df["score"] = np.where(sliding, -df["momentum"] * np.log1p(df["imp_per_day"]), 0.0).round(3)
df["reason_code"] = np.where(sliding, "early_slide_visible", "none")
df["action"] = np.where(sliding, "review_page", "monitor")

queue = (df.sort_values(["score", "imp_per_day"], ascending=False)
           .reset_index(drop=True))
queue.insert(0, "rank", queue.index + 1)
cols = ["rank", "client_hash_id", "content_hash_id", "content_type", "score", "reason_code", "action",
        "imp_per_day", "momentum", "age_days", "avg_position", "ctr"]
queue[cols].to_csv(OUT / "baseline_action_score.csv", index=False)
print(f"wrote work/outputs/baseline_action_score.csv ({len(queue):,} rows)")
print(f"flagged review_page: {sliding.sum():,} pages ({sliding.mean():.1%})")

def precision_at_k(k):
    return queue["is_declining"].head(k).mean()

metrics = {
    "slice": "March 2026, decision 2026-03-16, GSC available, >=100 impressions 1-15 Mar, created before March",
    "rows": int(len(df)), "base_rate": round(float(BASE_RATE), 4),
    "flagged_review_page": int(sliding.sum()),
    **{f"precision_at_{k}": round(float(precision_at_k(k)), 4) for k in (10, 50, 100, 500, 1000)},
    "signal_verdicts": {"staleness_age": "MIXED", "early_momentum": "CONFIRMED"},
    "inputs": INPUTS,
}
(OUT / "w04_baseline_metrics.json").write_text(json.dumps(metrics, indent=2))
pd.Series({k: v for k, v in metrics.items() if k.startswith("precision") or k == "base_rate"})


wrote work/outputs/baseline_action_score.csv (74,377 rows)
flagged review_page: 33,013 pages (44.4%)


base_rate            0.3359
precision_at_10      1.0000
precision_at_50      0.9800
precision_at_100     0.9600
precision_at_500     0.8320
precision_at_1000    0.7790
dtype: float64

The baseline is strong at the very top: Precision@10 = 1.0 and Precision@50 = 0.98, against a 33.6% base rate. It weakens as the list gets longer (Precision@500 = 0.83, Precision@1000 = 0.78). This is the number the week-5 model has to beat, and the fair comparison is at the larger K values, where the rule is no longer near-perfect.

## 3. Top-10 review

*For each of the top 10: action, why it's there, and what would make it wrong.*

In [5]:

top = queue.head(10).copy()
top["imp/day 1–7 Mar"]  = (top["imp_w1"].fillna(0) / 7).round(1)
top["imp/day 8–15 Mar"] = (top["imp_w2"].fillna(0) / 8).round(1)
top["client"] = top["client_hash_id"].str[:14] + "…"
top[["rank", "client", "content_type", "age_days", "imp/day 1–7 Mar", "imp/day 8–15 Mar",
     "momentum", "avg_position", "score", "reason_code", "action", "is_declining"]].round(2)


,rank,client,content_type,age_days,imp/day 1–7 Mar,imp/day 8–15 Mar,momentum,avg_position,score,reason_code,action,is_declining
0,1,client_73cda7b…,keyword article,227,11963.0,3.9,-7.81,0.11,67.35,early_slide_visible,review_page,1
1,2,client_0fa64a1…,keyword article,19,749.3,0.4,-6.30,9.29,36.93,early_slide_visible,review_page,1
2,3,client_0fa64a1…,keyword article,19,708.6,0.2,-6.34,8.08,36.81,early_slide_visible,review_page,1
3,4,client_20259bd…,keyword article,139,305.3,0.5,-5.32,27.63,26.42,early_slide_visible,review_page,1
4,5,client_62f4a7e…,keyword article,262,1670.4,40.9,-3.69,6.73,24.66,early_slide_visible,review_page,1
5,6,client_62f4a7e…,keyword article,262,442.4,5.6,-4.20,8.37,22.49,early_slide_visible,review_page,1
6,7,client_c182d11…,feedly article,231,200.7,1.0,-4.61,5.65,21.02,early_slide_visible,review_page,1
7,8,client_a80fca3…,keyword article,61,192.9,1.0,-4.57,8.50,20.66,early_slide_visible,review_page,1
8,9,client_20259bd…,keyword article,139,255.4,2.5,-4.29,3.83,20.61,early_slide_visible,review_page,1
9,10,client_e547b89…,keyword article,459,537.4,12.4,-3.70,0.30,20.52,early_slide_visible,review_page,1


All ten really declined in 16–31 March (`is_declining` = 1). But the *action* is a different question:

1. **`review_page`.** Why: the most visible page in the slice collapsed from ~11,963 to ~4 impressions/day within a week. Wrong if: the collapse is technical (page removed, de-indexed, tracking lost), not content decay; its average position of 0.11 is impossible in Google, so this row's data is suspect too.
2. **`review_page`.** Why: ~749 → 0.4 impressions/day. Wrong if: the page is only 19 days old, so this is a launch spike fading naturally, not a page that needs refreshing.
3. **`review_page`.** Why: ~709 → 0.25 impressions/day, same client and same age (19 days) as #2. Wrong if: same launch-spike story; two identical new pages dropping together points to a client-level event.
4. **`review_page`.** Why: ~305 → 0.5 impressions/day, then zero in late March. Wrong if: the page was unpublished or de-indexed; then the right action is a technical check, not a rewrite.
5. **`review_page`.** Why: ~1,670 → 41 impressions/day. Wrong if: the drop is site-wide: this client has 6 of the top 20 pages, all created the same day (262 days old). That looks like a migration or site-level change, not six stale pages.
6. **`review_page`.** Why: ~442 → 5.6 impressions/day, same client as #5. Wrong if: same site-level cause as #5.
7. **`review_page`.** Why: ~201 → 1 impression/day. Wrong if: it is a `feedly article` (news-style content), where fast decay is expected and a refresh adds little.
8. **`review_page`.** Why: ~193 → 1 impression/day. Wrong if: it was a temporary dip; it already recovered to ~8 impressions/day in late March.
9. **`review_page`.** Why: ~255 → 2.5 impressions/day, then zero. Wrong if: it is the same client as #4 with the same pattern, so a client-level event again.
10. **`review_page`.** Why: ~537 → 12 impressions/day on a 459-day-old page. Wrong if: its average position of 0.30 is impossible, so the position data (and possibly the impressions) for this page is unreliable.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks.** The rule finds pages that *will keep declining* very well, but the top of the list is dominated by **cliff drops**: impressions falling to near zero within one week. Those are more likely technical events (removal, de-indexing, a site migration, tracking loss) or launch spikes on very new pages (#2, #3) than the content decay that a "review/refresh" action is meant for. Three patterns show up:

- **Client clusters:** one client holds 6 of the top 20 (#5, #6 …), all created on the same day. That is a site-level cause, not page-level staleness.
- **Very new pages:** 19 days old (#2, #3). Their drop is natural settling after launch.
- **Impossible positions (< 1):** #1 and #10. That is a data-quality flag, not a signal.

What I would change later, not now, because the baseline stays frozen: route cliff drops to a separate `check_technical` action and add a minimum page age. The cliff pattern is also why Precision@10 is 1.0: a page that already went to zero almost always stays low, so the top of the list is the "easy" part.

**Leakage check.** The cell below confirms the score reads only first-half inputs, and that no future-window, label-derived or product-flag columns are used.

In [6]:

FUTURE_OR_LABEL = {"imp_h2", "is_declining"}
SNAPSHOT_AFTER_DECISION = {"content_updated_date", "last_optimized_date"}
PRODUCT_FLAGS = {"health_score", "priority_score", "action_type", "needs_ctr_fix", "is_quick_win", "refresh_tier"}

assert set(INPUTS).isdisjoint(FUTURE_OR_LABEL | SNAPSHOT_AFTER_DECISION | PRODUCT_FLAGS)
assert not PRODUCT_FLAGS & set(df.columns), "product flags are not in this data at all"
# recompute the score from INPUTS alone and confirm it matches
recomputed = np.where(df["momentum"] < -0.1, -df["momentum"] * np.log1p(df["imp_per_day"]), 0.0).round(3)
assert np.array_equal(recomputed, df["score"].values)
print("score uses only:", INPUTS, "| both built from 1–15 March days")
print("label (16–31 March) used only for precision@K | no snapshot dates | no product flags")

top20 = queue.head(20)
print(f"\ntop 20: {top20.client_hash_id.nunique()} clients, biggest client holds {top20.client_hash_id.value_counts().iloc[0]} rows")
print(f"top 20 pages under 30 days old: {(top20.age_days < 30).sum()} | with impossible position < 1: {(top20.avg_position < 1).sum()}")


score uses only: ['momentum', 'imp_per_day'] | both built from 1–15 March days
label (16–31 March) used only for precision@K | no snapshot dates | no product flags

top 20: 8 clients, biggest client holds 6 rows
top 20 pages under 30 days old: 4 | with impossible position < 1: 2


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.